# 🥩 Análisis del consumo de carne en Argentina — INECO (UADE)

Este notebook genera **todas las tablas y figuras** del informe de prensa sobre la
evolución del consumo de carne en Argentina y su comparación internacional,
**actualizadas a junio de 2026**.

- **Consumo por tipo de carne y comparación internacional** → se descargan en vivo
  desde la **API de la OCDE** (OECD-FAO Agricultural Outlook), anual, hasta 2026.
- **Precios (asado), IPC, salarios y exportaciones** → fuentes locales (IPCVA, INDEC, SIPA).

**Series mensuales extendidas a jun-2026:** el asado se completa con IPCVA (real),
el IPC con el INDEC Nacional (real) y, si faltara, con el REM; el salario se
**proyecta con media móvil** de la variación reciente (dic-2025 → jun-2026).

### Cómo usarlo
Ejecutá **Entorno de ejecución → Ejecutar todo** (`Ctrl+F9`). Al final se descarga un
`.zip` con las figuras en **600 dpi**, las tablas en **LaTeX** y un **Excel consolidado**
(`resultados_consumo_carne.xlsx`) con todas las series ordenadas.

## 1 · Preparación del entorno

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/santiagoriverti/analisis_consumo_carne.git'

if IN_COLAB:
    # Ruta FIJA y absoluta (idempotente: re-ejecutar no anida carpetas).
    REPO_DIR = '/content/analisis_consumo_carne'
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        # Ya clonado: traer SIEMPRE la última versión (evita correr código viejo).
        !cd "{REPO_DIR}" && git fetch -q --depth 1 origin main && git reset -q --hard origin/main
    else:
        !rm -rf "{REPO_DIR}"
        !git clone --depth 1 {REPO_URL} "{REPO_DIR}"
    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
else:
    # Local: subir hasta la raíz del repo (carpeta que contiene src/ y docs/).
    p = os.getcwd()
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, 'src')) and os.path.isdir(os.path.join(p, 'docs')):
            os.chdir(p); break
        p = os.path.dirname(p)

# Poner el directorio de trabajo al frente del path de importación.
if sys.path[:1] != [os.getcwd()]:
    sys.path.insert(0, os.getcwd())

print('Directorio de trabajo:', os.getcwd())

## 2 · Ejecutar el pipeline completo
Descarga la OCDE, carga las series, calcula y genera tablas + figuras.

In [ ]:
# Forzar re-importación de src por si quedó cacheado de una corrida anterior.
for _m in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]

from src import pipeline
from IPython.display import Image, display, Markdown

manifest = pipeline.run_all()
FIG = manifest['figuras']
TAB = manifest['tablas_df']

def mostrar(clave, width=1000):
    display(Image(filename=str(FIG[clave]), width=width))

## 3 · Tablas

Cada tabla se muestra como DataFrame y se imprime su versión **LaTeX** (guardada en
`outputs/tablas/*.tex`).

In [ ]:
# Tabla 1 — Mín./máx./promedio por tipo de carne (OCDE, 1990-2025)
display(Markdown('### Tabla 1 · Valores mínimo, máximo y promedio por tipo de carne'))
display(TAB['tabla1'])
print(open(manifest['tablas']['tabla1_min_max_prom'], encoding='utf-8').read())

In [ ]:
# Tabla 5 — Precio real del asado por gestión ($ de dic-2025)
display(Markdown('### Tabla 5 · Precio real del asado por gestión'))
display(TAB['tabla5'])
print(open(manifest['tablas']['tabla5_asado_real'], encoding='utf-8').read())

In [ ]:
# Tabla 7 — Esfuerzo salarial: kg de asado por salario
display(Markdown('### Tabla 7 · Kg de asado por salario, por gestión'))
display(TAB['tabla7'])
print(open(manifest['tablas']['tabla7_esfuerzo_salarial'], encoding='utf-8').read())

## 4 · Figuras — Consumo por tipo de carne (OCDE)

In [ ]:
display(Markdown('**Figura 2 · Evolución del consumo per cápita (kg) por tipo de carne**'))
mostrar('consumo_per_capita')

In [ ]:
display(Markdown('**Figura 3 · Consumo per cápita — Base 100 = 1990**'))
mostrar('consumo_base100')

In [ ]:
display(Markdown('**Figura 4 · Composición del consumo de carne (2026)**'))
mostrar('composicion_torta', width=650)

## 5 · Precios reales y esfuerzo salarial

In [ ]:
display(Markdown('**Figura 5 · Evolución del precio real del asado ($ de dic-2025, hasta jun-2026)**'))
mostrar('precio_asado_real')

In [ ]:
display(Markdown('**Figura 6 · Kg de asado que compra un salario (hasta jun-2026; salario proyectado desde ene-2026)**'))
mostrar('kg_asado_por_salario')

## 6 · Escenario internacional (carne vacuna)

In [ ]:
display(Markdown('**Figura 7a · Consumo vacuno internacional — absoluto (kg/hab)**'))
mostrar('vacuno_absoluto')
display(Markdown('**Figura 7b · Consumo vacuno internacional — Base 100 = 1990**'))
mostrar('vacuno_base100')

## 7 · Exportaciones y precios relativos

> **Nota:** las exportaciones absolutas (M kg / M USD) son **anuales** y la fuente
> (`analisis_carne.xlsx`) llega a **2025**. 2026 es un año en curso (incompleto), por lo
> que no se grafica como barra anual. La versión de **índices INDEC base 2004=100** sí
> llega a jun-2026 (se muestra más abajo).

In [ ]:
_ult = int(manifest['dataframes']['exportaciones']['anio'].max())
display(Markdown(f'**Figura 8a · Exportaciones de carne bovina — volumen (M kg, anual, hasta {_ult})**'))
mostrar('exportaciones_kg')
display(Markdown(f'**Figura 8b · Exportaciones de carne bovina — valor (miles de M USD, anual, hasta {_ult})**'))
mostrar('exportaciones_usd')

In [ ]:
# Versión alternativa de exportaciones: índices INDEC base 2004 = 100
if 'exportaciones_indices' in FIG:
    display(Markdown('**Alternativa · Exportaciones — índices INDEC (base 2004 = 100)**'))
    mostrar('exportaciones_indices')

In [ ]:
display(Markdown('**Figura 9 · Precio relativo: kg de pollo por 1 kg de asado**'))
mostrar('relativos')

## 8 · Descargar todos los resultados
Genera un `.zip` con `outputs/`: figuras (600 dpi), tablas LaTeX y el Excel consolidado `resultados_consumo_carne.xlsx`.

In [ ]:
import shutil
from src import config as C

# Usar rutas ABSOLUTAS (C.OUTPUTS) para no depender del directorio de trabajo.
zip_path = shutil.make_archive(str(C.ROOT / 'resultados_consumo_carne'), 'zip', str(C.OUTPUTS))
print('ZIP generado:', zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
else:
    print('Resultados disponibles en:', C.OUTPUTS)